![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [1]:
# QuantBook Analysis Tool 
# For more information see [https://www.quantconnect.com/docs/v2/our-platform/research/getting-started]
qb = QuantBook()
spy = qb.add_equity("SPY")
# Locally Lean installs free sample data, to download more data please visit https://www.quantconnect.com/docs/v2/lean-cli/datasets/downloading-data 
qb.set_start_date(2021, 4, 1)
history = qb.history(qb.securities.keys(), 360, Resolution.DAILY)

# Indicator Analysis
bbdf = qb.indicator(BollingerBands(30, 2), spy.symbol, 360, Resolution.DAILY)
bbdf.drop('standarddeviation', axis=1).plot()

Importing and global objects

In [2]:
# QuantBook-compatible adaptation of the class logic
# Remove QCAlgorithm and replace with QuantBook usage
# Create functions you can call in cells interactively

from AlgorithmImports import *
from datetime import timedelta, datetime
import pandas as pd
from math import sqrt

# Global storage dictionary
OBJECT_LOG = {}
earnings_date = datetime(2021, 4, 28)

def describe_object(obj, obj_name="unknown"):
    description = {
        "name": obj_name,
        "type": str(type(obj)),
        "attributes": {},
        "methods": []
    }

    for attr in dir(obj):
        if attr.startswith("__"):
            continue
        try:
            # value = getattr(obj, attr)
            value = None
            if callable(value):
                description["methods"].append(attr)
            else:
                description["attributes"][attr] = value
        except Exception as e:
            description["attributes"][attr] = f"<Error reading: {e}>"

    OBJECT_LOG[obj_name] = description


Definicion de un universo de 3 acciones

In [3]:
qb = QuantBook()

# Example: set universe manually
tickers = ["AAPL", "MSFT", "GOOG"]
symbols = [qb.add_equity(t, Resolution.DAILY).Symbol for t in tickers]

Definicion de las Funciones 1 funcion por bloque

In [4]:
# Example: get option chains around a date
def get_two_closest_option_chains(symbol, earnings_time):
    lookback_time = earnings_time - timedelta(days=30)
    frontlook_time = earnings_time + timedelta(days=30)
    contracts = qb.option_chain_provider.get_option_contract_list(symbol, lookback_time)
    if not contracts:
        return [], []

    contracts_by_exp = {}
    for c in contracts:
        exp = c.ID.Date
        contracts_by_exp.setdefault(exp, []).append(c)

    before = [exp for exp in contracts_by_exp if earnings_time > exp >= lookback_time]
    after_or_equal = [exp for exp in contracts_by_exp if earnings_time <= exp < frontlook_time]

    near = sorted(before)[-1] if before else None
    far = sorted(after_or_equal)[0] if after_or_equal else None

    return contracts_by_exp.get(near, []), contracts_by_exp.get(far, [])

Get Option chains near the earnings_date

In [5]:
for step in symbols:
    var_near_chain, var_far_chain = get_two_closest_option_chains(step, earnings_date)

print(f'This function is unpacked so that you can get 2 chains, a near {var_near_chain}, and a far {var_far_chain} ')
print(f'Each chain is a list of symbols (each being a contract) {var_near_chain[0:2]}, {var_far_chain[0:2]}, etc...')
print(f'Each chain got the following values as per the last example {var_near_chain[0].value}, {var_near_chain[1].value}, etc...')



In [6]:
# Example: match contracts by strike and type
def match_option_contracts_by_strike_and_type(near_chain, next_chain):
    matches = []
    next_lookup = {(c.ID.OptionRight, c.ID.StrikePrice): c for c in next_chain}

    for near_contract in near_chain:
        key = (near_contract.ID.OptionRight, near_contract.ID.StrikePrice)
        if key in next_lookup:
            matches.append((near_contract, next_lookup[key]))
    return matches

Here we match the options in each case, returning again an object

In [7]:
var_matched_options = match_option_contracts_by_strike_and_type(var_near_chain, var_far_chain)
print(f"var_matched_options returns tuples being on the same Strike, Type BUT different expiration, one before and another after the specified earnings_date {earnings_date}")
print(f"near {var_matched_options[0][0].value} -- {earnings_date} -- far {var_matched_options[0][1].value}")
dir_per_matched_options_0_0 = dir(var_matched_options[0][0])


From here we start calculating the IV slope

First we show here a portion of the tupled options

In [8]:
for step_tuple_near, step_tuple_far in var_matched_options:
    print(qb.add_optionContract(step_tuple_near, Resolution.DAILY), '-', qb.add_optionContract(step_tuple_far, Resolution.DAILY))
    

Second we calculate the IV for each of the tupled matched contracts

In [9]:
def get_implied_volatilities(near_symbol, far_symbol):
    def compute_iv(symbol):
        start_date = earnings_date - timedelta(days=30)
        current_date = earnings_date
        end_date = earnings_date + timedelta(days=30)
        contract = qb.add_optionContract(symbol, Resolution.DAILY)
        history = qb.history(contract.symbol, symbol.ID.Date - timedelta(15), symbol.ID.Date - timedelta(2), Resolution.DAILY)
        if history.empty:
            print(f"No history for {symbol}")
            return None

        last_row = history.iloc[-1]
        try:
            underlying_symbol = symbol.Underlying
            underlying_history = qb.history([underlying_symbol], start_date, current_date, Resolution.DAILY)
            if underlying_history.empty:
                print(f"No underlying history for {underlying_symbol}")
                return None

            S = underlying_history.iloc[-1].close
            K = symbol.ID.StrikePrice
            expiry = symbol.ID.Date
            T = (expiry - last_row.name[-1]).days / 365.0
            if T <= 0:
                print(f"Non-positive T for {symbol}")
                return None

            price = (last_row.askclose + last_row.bidclose)/2.0
            if price <=0:
                print(f"Non-positive price for {symbol}")
                return None

            iv = black_scholes_call_iv(S, K, T, 0.0, price)
            print(f"Computed IV for {symbol}: {iv}")
            return iv
        except Exception as e:
            print(f"Error computing IV for {symbol}: {e}")
            return None

    near_iv = compute_iv(near_symbol)
    far_iv = compute_iv(far_symbol)

    return (near_iv, far_iv)

Here is the output of the implied_volatilies, 

Some Debug

In [10]:
aapl = qb.add_equity("AAPL", Resolution.DAILY).Symbol
contracts = qb.option_chain_provider.get_option_contract_list(aapl, datetime(2024, 4, 1))

# sort by expiry and strike
contracts = sorted(contracts, key=lambda x: (x.ID.Date, abs(x.ID.StrikePrice - 130)))  # AAPL ~130

for contract in contracts[:5]:  # test top 5
    print(f"Trying contract: {contract.Value}, Expiry: {contract.ID.Date}, Strike: {contract.ID.StrikePrice}")
    qb.add_optionContract(contract, Resolution.DAILY)
    df = qb.history([contract], 10, Resolution.DAILY)
    print(f"→ Rows: {len(df)}")
    if not df.empty:
        print(df.head())
        break

More Debug

In [15]:
from QuantConnect import Resolution
from datetime import datetime, timedelta

# Get the underlying symbol
underlying = qb.add_equity("AAPL", Resolution.MINUTE).Symbol

# Get the option chain
option = qb.add_option("AAPL")
option.SetFilter(-2, 2, 0, 30)  # Example filter: +/-2 strikes, 0-30 days

# Define date range
start = datetime(2024, 4, 1)
end = datetime(2024, 4, 30)

# Warm up the option chain
qb.history(underlying, start, end, Resolution.MINUTE)

# Get the available option contracts
chains = qb.option_chain_provider.get_option_contract_list(underlying, start)

# Pick one contract manually (example: first one)
contract_symbol = chains[0]

# Add the option contract to QuantBook
contract = qb.add_optionContract(contract_symbol, Resolution.DAILY)

# Get historical price data
history = qb.history(contract.Symbol, start, end, Resolution.DAILY)

# Show the result
history.head()


In [24]:
valid_iv_results = []
for step_tuple_near, step_tuple_far in var_matched_options:
    near_iv, far_iv = get_implied_volatilities(step_tuple_near, step_tuple_far)
    print(f'near_iv {near_iv} , far_iv {far_iv} ')
    if near_iv and far_iv:
        if near_iv > far_iv:
            print(f'valid case:  near_iv {near_iv} bigger than far_iv {far_iv} ')
        valid_iv_results.append((near_iv, far_iv))

In [17]:
# Function to update an option contract with its historical prices
from QuantConnect.Data.Market import TradeBar
from datetime import datetime


def update_option_with_history(symbol):
    contract = qb.add_optionContract(symbol, Resolution.DAILY)

    # Use expiry as reference and fetch a wide window around it
    start = symbol.ID.Date - timedelta(days=30)
    end = symbol.ID.Date + timedelta(days=1)

    try:
        history = qb.history([symbol], start, end, Resolution.DAILY)
        if history.empty:
            print(f"No history for {symbol} between {start} and {end}")
            return False

        for idx, row in history.iterrows():
            timestamp = idx[1] if isinstance(idx, tuple) else idx
            if isinstance(timestamp, pd.Timestamp):
                timestamp = timestamp.to_pydatetime()
            trade_bar = TradeBar(
                timestamp,
                symbol,
                float(row['open']),
                float(row['high']),
                float(row['low']),
                float(row['close']),
                float(row['volume']),
                timedelta(days=1)
            )
            contract.Update(trade_bar)

        print(f"Updated {symbol.Value} with {len(history)} bars.")
        return True
    except Exception as e:
        print(f"Failed to update history for {symbol.Value}: {e}")
        return False

# Example usage:
# update_option_with_history(your_option_symbol)
# Then call get_implied_volatilities()

In [16]:
def black_scholes_call_iv(S, K, T, r, option_price):
    if T <= 0:
        print(f"[IV] Skipping: T <= 0 (T={T:.6f})")
        return None

    intrinsic_value = max(S - K * exp(-r * T), 0)
    if option_price <= intrinsic_value:
        print(f"[IV] Skipping: option_price <= intrinsic_value ({option_price:.4f} <= {intrinsic_value:.4f})")
        return None

    MAX_ITER = 100
    PRECISION = 1.0e-5
    sigma = 0.2

    for i in range(MAX_ITER):
        try:
            d1 = (log(S / K) + (r + sigma**2 / 2) * T) / (sigma * sqrt(T))
            d2 = d1 - sigma * sqrt(T)
            price = S * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
            vega = S * norm.pdf(d1) * sqrt(T)

            if vega < 1e-8:
                print(f"[IV] Skipping: vega too small (vega={vega:.8f}) at iter {i}")
                return None

            diff = price - option_price
            print(f"[IV] iter {i}: sigma={sigma:.6f}, price={price:.4f}, target={option_price:.4f}, diff={diff:.6f}")

            if abs(diff) < PRECISION:
                print(f"[IV] Converged at iter {i} with sigma={sigma:.6f}")
                return sigma

            sigma -= diff / vega
        except Exception as e:
            print(f"[IV] Error at iter {i}: {e}")
            return None

    print(f"[IV] Failed to converge after {MAX_ITER} iterations")
    return None


In [23]:
print(valid_iv_results)



In [29]:
# Example: calculate IV slope for one symbol
def calculate_iv_slope(symbol, earnings_date):
    front_chain, back_chain = get_two_closest_option_chains(symbol, earnings_date)
    contracts_tuples = match_option_contracts_by_strike_and_type(front_chain, back_chain)
    # debug_option_history([contracts_tuples[0][0], contracts_tuples[0][1]], datetime(2021,4,1), datetime(2021,4,30))
    slopes = []
    for front, back in contracts_tuples:
        qb.add_optionContract(front, Resolution.DAILY)
        qb.add_optionContract(back, Resolution.DAILY)

        near_iv, far_iv = get_implied_volatilities(front, back)

        diff_days = (back.ID.Date.date() - front.ID.Date.date()).days
        if near_iv is None or far_iv is None:
            print(f"Skipping pair due to missing IVs: Near={near_iv}, Far={far_iv}")
            continue
        if diff_days <= 0:
            print(f"Skipping pair due to non-positive diff_days: {diff_days}")
            continue

        # print(f"Computed IVs: Near={near_iv}, Far={far_iv}, Diff Days={diff_days}")

        slope = (near_iv - far_iv) / diff_days
        slopes.append({
            "strike": front.ID.StrikePrice,
            "near_contract": front.value,
            "far_contract": back.value,
            "slope": slope,
            "near_iv": near_iv,
            "far_iv": far_iv,
            "diff_days": diff_days
        })
    return slopes

Ejemplo de Uso 

In [30]:
# Example usage:

slopes = calculate_iv_slope(symbols[0], earnings_date)
print(symbols[0].value)
print(slopes)

Debugging option_history definition and launching

In [31]:
for i in slopes:    
    print(f'{i}')

In [ ]:
def debug_option_history(symbols, start, end):
    for s in symbols:
        try:
            print(f"Requesting history for {s.Value} from {start} to {end}")
            history = qb.history([s], start, end, Resolution.DAILY)
            if history.empty:
                print(f"EMPTY history for {s.Value}")
            else:
                print(f"History rows for {s.Value}: {len(history)}")
                print(history)
        except Exception as e:
            print(f"Error fetching history for {s.Value}: {e}")

# Example usage:
# debug_option_history([symbols[0], symbols[1]], datetime(2021,4,1), datetime(2021,4,30))

More Debugging in a deeper manner

In [ ]:
# Deep Debug approach to understand why you have no history
def deep_debug_option_history(symbol):
    print(f"==== Debugging {symbol.Value} ====")
    
    # Check if the security exists in qb.Securities
    if symbol in qb.Securities:
        sec = qb.Securities[symbol]
        print(f"Security found: {sec.Symbol.Value}")
        print(f"Exchange Time Zone: {sec.Exchange.TimeZone}")
        print(f"Expiry: {symbol.ID.Date}")
        print(f"Strike: {symbol.ID.StrikePrice}")
        print(f"OptionRight: {symbol.ID.OptionRight}")
        print(f"Underlying: {symbol.Underlying}")
    else:
        print("Security not yet added to qb.Securities")

    # Skip GetLastKnownPrice; not supported for options

    # Check OptionContract list
    contracts = qb.option_chain_provider.get_option_contract_list(symbol.Underlying, symbol.ID.Date)
    print(f"Option contracts returned for expiry date {symbol.ID.Date}: {len(contracts)}")
    for c in contracts:
        print(f"Contract: {c.Value}")

    # Fetch history for a wide range
    start = symbol.ID.Date - timedelta(days=30)
    end = symbol.ID.Date + timedelta(days=1)
    try:
        history = qb.history([symbol], start, end, Resolution.DAILY)
        if history.empty:
            print("EMPTY history")
        else:
            print(f"History rows: {len(history)}")
            print(history)
    except Exception as e:
        print(f"Error fetching history: {e}")


# Example usage:
# deep_debug_option_history(your_option_symbol)